# Chapter 1 — Common Conventions and API Elements of scikit-learn

> Reproduction notebook for **scikit-learn Cookbook, Third Edition** (John Sukup, Packt 2025).
> Each section contains: (1) a short **theory explanation**, (2) the **code reproduced from the book**, and (3) a short **takeaway**.

## Chapter Summary

Chapter 1 does not build a real model yet. Instead it explains the *grammar* of scikit-learn so that every later chapter feels familiar.

| Idea | One-line summary |
|---|---|
| Design philosophy | Consistency, simplicity, modularity and reusability: all models share one interface. |
| Estimators | Objects that learn from data via `fit()` and predict via `predict()`. |
| Transformers | Objects that change data via `fit()` + `transform()` (scaling, encoding, ...). |
| Custom objects | Subclass `BaseEstimator` (+ mixins) to plug your own logic into pipelines and grid search. |
| Pipelines | Chain preprocessing and a model into a single, reproducible object (links to MLOps). |
| Attributes & methods | `coef_`, `intercept_`, `score()`, `get_params()`, `set_params()` are shared across models. |
| Hyperparameter search | `GridSearchCV`, `RandomizedSearchCV` and the successive-halving variants. |
| Metadata | Estimator tags and *metadata routing* (routers send metadata to consumers). |
| Best practices | Uniform API, preprocess first, use pipelines, cross-validate, tune hyperparameters. |

## 1. scikit-learn's design philosophy

**Theory.** scikit-learn is built on four principles:

* **Consistency** – nearly every algorithm exposes the same verbs (`fit`, `predict`, `transform`). Swapping `LinearRegression` for `RandomForestRegressor` changes one line of code.
* **Simplicity** – sensible defaults, so a first model needs only a few lines.
* **Modularity** – estimators, transformers and pipelines are small building blocks that can be combined.
* **Reusability** – because every block follows the same contract, you can write your own and reuse it across projects.

A nice example is the `Pipeline` class: preprocessing (scaling, encoding) and modelling are packed in one object, which makes the workflow reproducible.

> Spelling note from the book: the library name is always written in lowercase, **scikit-learn** (pronounced *sy-kit*, "science kit").

## 2. Understanding estimators

**Theory.** An *estimator* is any object that learns from data. Two essential methods:

* `fit(X, y)` – learns parameters from training data (for example, the coefficients of a line).
* `predict(X_new)` – uses the learned parameters to predict new data.

`fit_predict()` combines both and is mostly used in **unsupervised learning** (e.g. `KMeans`), where the data used for training is also the data we want labels for. In supervised learning we usually `fit` on the training set and `predict` on a hold-out set.

In [1]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Predict values for new data
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print(predictions)

[5.75 6.7 ]


In [2]:
# fit_predict() example with a clustering estimator
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=2024, n_init=10)
labels = kmeans.fit_predict(X)
print(labels)

[0 0 0 1 1]


**Takeaway.** Whether the model is a simple line or a random forest, the pattern is always *create → fit → predict*.

## 3. Transformers and the `transform()` method

**Theory.** A *transformer* modifies data (scaling, normalisation, encoding). It follows the same contract:

* `fit()` learns what is needed from the data (e.g. mean and standard deviation for `StandardScaler`).
* `transform()` applies the learned transformation.
* `fit_transform()` does both in one call.

**Important rule (avoid data leakage):** call `fit_transform()` on the **training** data only. For the **test** data use `transform()` only. Re-fitting on test data would use a *different* mean/standard deviation and make the model unreliable in production.

`StandardScaler` computes the z-score: $z = (x-\mu)/\sigma$.

In [3]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()
scaler.fit(X)                    # learn mean and std
X_scaled = scaler.transform(X)   # apply them
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [4]:
# The same result in a single step with fit_transform()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)
print("learned mean:", scaler.mean_, "| learned std:", scaler.scale_)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
learned mean: [3. 4.] | learned std: [1.63299316 1.63299316]


## 4. Handling custom estimators and transformers

**Theory.** scikit-learn is extensible. To create your own component:

1. Subclass `BaseEstimator` (gives `get_params` / `set_params` for free).
2. Add a **mixin** such as `TransformerMixin` (gives `fit_transform`) or `ClassifierMixin` / `RegressorMixin` (gives `score`).
3. Implement `fit()` and `transform()` (transformers) or `fit()` and `predict()` (models).
4. Use `check_is_fitted()` to make sure `fit()` was called first.

A *mixin* is a small class that adds reusable functionality to other classes without a deep inheritance tree.

*The book only describes this idea in text; the cell below is a small illustration added for the explanation.*

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class LogTransformer(BaseEstimator, TransformerMixin):
    """Custom transformer: applies log(1 + x) to every feature."""

    def __init__(self, offset=0.0):
        self.offset = offset            # a hyperparameter

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]  # learned attribute (ends with _)
        return self

    def transform(self, X):
        check_is_fitted(self, "n_features_in_")
        return np.log1p(X + self.offset)


lt = LogTransformer()
print(lt.fit_transform(np.array([[0.0, 1.0], [9.0, 99.0]])))
print(lt.get_params())      # inherited from BaseEstimator

[[0.         0.69314718]
 [2.30258509 4.60517019]]
{'offset': 0.0}


## 5. Pipelines and workflow automation

**Theory.** A `Pipeline` chains transformers and (optionally) a final estimator:

`raw data → transformer 1 → transformer 2 → ... → estimator`

Benefits: steps always run in the right order, the same transformations are applied to train and test data, and the whole object can be tuned or saved at once.

**MLOps** (machine-learning operations) extends this idea to the whole life cycle: automating development, testing, deployment and monitoring so models stay reliable after release. The book stresses that *"there is no such thing as a model that works well forever"*. scikit-learn supports MLOps through `Pipeline`, `GridSearchCV`, and persistence libraries (`joblib`, `pickle`), and integrates with platforms like MLflow or Kubeflow.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=300, n_features=6, random_state=2024)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression()),
])
pipe.fit(X_train, y_train)                 # scaler.fit_transform -> clf.fit
print("test accuracy:", pipe.score(X_test, y_test))  # scaler.transform -> clf.predict

test accuracy: 0.8833333333333333


## 6. Common attributes and methods

**Theory.** Fitted models expose shared attributes. By convention, attributes **learned from data end with an underscore**:

* `coef_` and `intercept_` – learned coefficients of linear models (useful for interpretation).
* `score(X, y)` – default metric: **accuracy** for classifiers, **R²** for regressors.

> Note from the book: R² always increases when more variables are added, so *adjusted R²* is often preferred because it penalises the number of variables.

In [7]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Model R-squared:", model.score(X, y))

Coefficients: [0.95]
Intercept: 0.04999999999999938
Model R-squared: 0.9809782608695652


## 7. Hyperparameter tuning with search methods

**Theory.** *Parameters* are learned by the model (`coef_`); *hyperparameters* are set by us **before** training and control how the model learns (e.g. `max_depth`).

* `GridSearchCV` – tries **every** combination in a grid.
* `RandomizedSearchCV` – samples random combinations (cheaper for large spaces).
* Successive halving variants – start with many candidates on little data and keep only the best.
* `get_params()` / `set_params()` – read / manually change hyperparameters.

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.set_params(n_estimators=100, max_depth=10, random_state=42)
print(model.get_params())

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [9]:
# Using a search method on top of get/set_params (small example)
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={"n_estimators": [20, 50], "max_depth": [3, 6]},
    cv=3,
)
grid.fit(X_train, y_train)
print(grid.best_params_, round(grid.best_score_, 3))

{'max_depth': 6, 'n_estimators': 50} 0.929


## 8. Working with metadata: tags and more

**Theory.**

* **Estimator tags** describe capabilities of an estimator (e.g. whether it supports multi-output data or missing values) so scikit-learn can adapt cross-validation and pipeline behaviour.
* **Metadata routing** controls how extra information (sample weights, group labels, fit parameters) travels through a pipeline. There are two roles: **routers** (pass metadata on) and **consumers** (use it in calculations).
* Example from the book: in an imbalanced-data project, pass `sample_weight` only to the classifier and ignore it in the scaler.

The cell below shows the simplest, always-available way to send `sample_weight` to one pipeline step (the `step__param` naming convention).

In [10]:
weights = np.where(y_train == 1, 2.0, 1.0)      # give class 1 twice the weight

pipe = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression())])
pipe.fit(X_train, y_train, clf__sample_weight=weights)   # only the classifier consumes it
print("accuracy with sample weights:", round(pipe.score(X_test, y_test), 3))

accuracy with sample weights: 0.867


## 9. Best practices for API usage

* **Uniform API** – rely on `fit` / `predict` / `transform`.
* **Preprocess data** with `sklearn.preprocessing` (scaling, encoding, missing values) before modelling.
* **Use pipelines** for multi-step workflows; this also simplifies tuning.
* **Cross-validate** (`sklearn.model_selection`) to estimate generalisation.
* **Tune hyperparameters** with `GridSearchCV` / `RandomizedSearchCV`.

---
### Chapter 1 key takeaways
1. All scikit-learn objects speak the same small language: `fit`, `predict`, `transform`.
2. Fit on training data only; transform test data.
3. Pipelines + search methods make workflows modular, reproducible and ready for production.